In [ ]:

import os, gc, json, math, random, re, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
import timm
import torchaudio
warnings.filterwarnings('ignore')

CFG = {
    'COMP_DIR':    '/kaggle/input/birdclef-2026',
    'PSEUDO_PARQUET': None,
    'OUT_DIR':     '/kaggle/working',
    'BACKBONE':    'tf_efficientnet_b0.ns_jft_in1k',
    'SR':          32000,
    'CLIP_SEC':    5.0,
    'N_FFT':       2048,
    'HOP':         512,
    'N_MELS':      128,
    'FMIN':        50,
    'FMAX':        14000,
    'EPOCHS':      20,
    'BATCH_SIZE':  32,
    'LR':          3e-4,
    'WD':          1e-4,
    'MIXUP_ALPHA': 0.5,
    'FOLDS_TO_RUN':[0],
    'N_FOLDS':     5,
    'SEED':        42,
    'NUM_WORKERS': 4,
    'MAX_SAMPLES_PER_CLASS': 500,
    'MIN_SAMPLES_PER_CLASS': 5,
    'DROP_PATH_RATE': 0.15,
    'SOUNDSCAPE_FRAC': 0.7,
    'PSEUDO_LABEL_SMOOTH_MIN': 0.0,
}

os.makedirs(CFG['OUT_DIR'], exist_ok=True)
random.seed(CFG['SEED']); np.random.seed(CFG['SEED']); torch.manual_seed(CFG['SEED'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

BASE = Path(CFG['COMP_DIR'])
train_csv = pd.read_csv(BASE / 'train.csv')
sample_sub = pd.read_csv(BASE / 'sample_submission.csv')
CLASSES = sample_sub.columns[1:].tolist()
N_CLASSES = len(CLASSES)
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
print(f'Classes: {N_CLASSES}, train rows: {len(train_csv)}')

COL_FILE   = 'filename'
COL_PRIM   = 'primary_label'
COL_SECOND = 'secondary_labels'


In [ ]:
_pseudo_path = None
_candidates = [
    '/kaggle/input/soundscape-pseudo-labels/soundscape_pseudo_labels.parquet',
    '/kaggle/input/birdclef2026-soundscape-pseudo/soundscape_pseudo_labels.parquet',
    '/kaggle/working/soundscape_pseudo_labels.parquet',
]
for c in _candidates:
    if Path(c).exists():
        _pseudo_path = c
        break
if _pseudo_path is None:
    _matches = list(Path('/kaggle/input').rglob('soundscape_pseudo*.parquet'))
    if _matches:
        _pseudo_path = str(_matches[0])
assert _pseudo_path, ('soundscape_pseudo_labels.parquet not found. '
                      'Run make_soundscape_pseudo_labels.ipynb first and attach as dataset.')
CFG['PSEUDO_PARQUET'] = _pseudo_path
print(f'Pseudo-labels: {_pseudo_path}')

pseudo_df = pd.read_parquet(_pseudo_path)
print(f'Pseudo rows: {len(pseudo_df)}  (~{len(pseudo_df) // 12} unique soundscape files)')
assert 'filename' in pseudo_df.columns and 'start_sec' in pseudo_df.columns and 'labels' in pseudo_df.columns
_lab0 = np.array(pseudo_df.iloc[0]['labels'], dtype=np.float32)
assert _lab0.shape == (N_CLASSES,), f'Pseudo label dim {_lab0.shape} != ({N_CLASSES},)'
print(f'Pseudo label dim OK: {_lab0.shape}')


In [ ]:
def balance_index(df):
    rows = []
    for cls, sub in df.groupby(COL_PRIM):
        if len(sub) > CFG['MAX_SAMPLES_PER_CLASS']:
            sub = sub.sample(CFG['MAX_SAMPLES_PER_CLASS'], random_state=CFG['SEED'])
        if len(sub) < CFG['MIN_SAMPLES_PER_CLASS']:
            reps = math.ceil(CFG['MIN_SAMPLES_PER_CLASS'] / max(1, len(sub)))
            sub = pd.concat([sub] * reps, ignore_index=True).head(CFG['MIN_SAMPLES_PER_CLASS'])
        rows.append(sub)
    return pd.concat(rows, ignore_index=True)

train_csv = train_csv[train_csv[COL_PRIM].isin(set(CLASSES))].reset_index(drop=True)
print('Filtered focal size:', len(train_csv))

def detect_group_col(df):
    for c in ('author', 'recording_id'):
        if c in df.columns and df[c].astype(str).str.len().gt(0).any():
            return c
    return COL_FILE
GROUP_COL = detect_group_col(train_csv)
print(f'CV grouping key: {GROUP_COL}')
groups = train_csv[GROUP_COL].astype(str).fillna('')
groups = groups.where(groups != '', train_csv[COL_FILE].astype(str)).str.lower().values
sgkf = StratifiedGroupKFold(n_splits=CFG['N_FOLDS'], shuffle=True, random_state=CFG['SEED'])
y_strat = train_csv[COL_PRIM].values
folds = list(sgkf.split(train_csv, y_strat, groups=groups))
for f, (tr, va) in enumerate(folds):
    overlap = set(groups[tr].tolist()) & set(groups[va].tolist())
    assert not overlap, f'Fold {f}: {len(overlap)} groups leak across train/val'
print(f'Folds: {len(folds)} (running {CFG["FOLDS_TO_RUN"]}, no group overlap)')


In [ ]:
CLIP_SAMPLES = int(CFG['SR'] * CFG['CLIP_SEC'])

_MEL = torchaudio.transforms.MelSpectrogram(
    sample_rate=CFG['SR'], n_fft=CFG['N_FFT'], hop_length=CFG['HOP'],
    n_mels=CFG['N_MELS'], f_min=CFG['FMIN'], f_max=CFG['FMAX'], power=2.0)
_DB = torchaudio.transforms.AmplitudeToDB(top_db=80)

_IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def wav_to_mel_3ch(wav_np):
    x = torch.from_numpy(wav_np)
    with torch.no_grad():
        m = _DB(_MEL(x))
        m = (m - m.mean()) / (m.std() + 1e-6)
    m = m.unsqueeze(0).expand(3, -1, -1)
    m = (m - _IMAGENET_MEAN) / _IMAGENET_STD
    return m


In [ ]:
def _shift_zero_pad(x, shift):
    if shift == 0: return x
    out = np.zeros_like(x)
    if shift > 0:
        out[shift:] = x[:len(x) - shift]
    else:
        out[:len(x) + shift] = x[-shift:]
    return out

class PinkNoise:
    def __init__(self, prob=0.3, snr_db=(5, 20)):
        self.prob, self.snr_db = prob, snr_db
    def __call__(self, x):
        if random.random() > self.prob: return x
        snr = random.uniform(*self.snr_db)
        n = np.cumsum(np.random.randn(len(x)).astype(np.float32))
        n /= (np.abs(n).max() + 1e-6)
        sig_p = (x ** 2).mean() + 1e-9
        noise_p = sig_p / (10 ** (snr / 10))
        return x + n * math.sqrt(noise_p)

class ExternalNoise:
    def __init__(self, noise_dir, prob=0.4, snr_db=(0, 15)):
        self.files = []
        if noise_dir is not None:
            p = Path(noise_dir)
            if p.exists():
                self.files = sorted([*p.rglob('*.ogg'), *p.rglob('*.wav'),
                                     *p.rglob('*.flac'), *p.rglob('*.mp3')])
        self.prob = prob if self.files else 0.0
        self.snr_db = snr_db
    def __call__(self, x):
        if random.random() > self.prob or not self.files: return x
        sc_path = random.choice(self.files)
        try:
            with sf.SoundFile(str(sc_path)) as f:
                start = random.randint(0, max(0, f.frames - len(x)))
                f.seek(start)
                noise = f.read(len(x), dtype='float32', always_2d=False)
                if noise.ndim == 2: noise = noise.mean(axis=1)
            if len(noise) < len(x): noise = np.pad(noise, (0, len(x) - len(noise)))
            noise = noise[:len(x)]
            snr = random.uniform(*self.snr_db)
            sp = (x**2).mean() + 1e-9
            np_pwr = sp / (10 ** (snr/10))
            return x + noise * math.sqrt(np_pwr / ((noise**2).mean() + 1e-9))
        except Exception:
            return x

def spec_augment(spec, time_mask=40, freq_mask=20, n_time=2, n_freq=2, prob=0.8):
    if random.random() > prob: return spec
    B, _, F_, T = spec.shape
    for _ in range(n_time):
        t = random.randint(0, time_mask)
        if t == 0 or t >= T: continue
        s = random.randint(0, T - t)
        spec[:, :, :, s:s+t] = 0.0
    for _ in range(n_freq):
        f = random.randint(0, freq_mask)
        if f == 0 or f >= F_: continue
        s = random.randint(0, F_ - f)
        spec[:, :, s:s+f, :] = 0.0
    return spec


In [ ]:

class MixedSEDDataset(Dataset):
    def __init__(self, focal_df, pseudo_df, train_audio_dir, soundscape_dir,
                 is_train=True):
        self.focal_df = focal_df.reset_index(drop=True)
        self.pseudo_df = pseudo_df.reset_index(drop=True) if pseudo_df is not None else None
        self.train_audio_dir = Path(train_audio_dir)
        self.soundscape_dir = Path(soundscape_dir)
        self.is_train = is_train
        self.soundscape_frac = CFG['SOUNDSCAPE_FRAC'] if (is_train and self.pseudo_df is not None) else 0.0
        self.pink = PinkNoise(prob=0.3 if is_train else 0.0)
        self.bg_noise = ExternalNoise(CFG.get('EXTERNAL_NOISE_DIR'),
                                      prob=0.4 if is_train else 0.0)

    def __len__(self):
        if self.pseudo_df is None or not self.is_train:
            return len(self.focal_df)
        return max(len(self.focal_df), len(self.pseudo_df))

    def _load_focal_5s(self, path):
        try:
            with sf.SoundFile(str(path)) as f:
                total = f.frames
                if total >= CLIP_SAMPLES:
                    start = random.randint(0, total - CLIP_SAMPLES) if self.is_train else 0
                    f.seek(start)
                    wav = f.read(CLIP_SAMPLES, dtype='float32', always_2d=False)
                else:
                    wav = f.read(dtype='float32', always_2d=False)
            if wav.ndim == 2: wav = wav.mean(axis=1)
            wav = wav.astype(np.float32, copy=False)
            if len(wav) > CLIP_SAMPLES: wav = wav[:CLIP_SAMPLES]
            elif len(wav) < CLIP_SAMPLES: wav = np.pad(wav, (0, CLIP_SAMPLES - len(wav)))
            return wav
        except Exception:
            return np.zeros(CLIP_SAMPLES, dtype=np.float32)

    def _load_soundscape_window(self, path, start_sec):
        try:
            with sf.SoundFile(str(path)) as f:
                start = int(start_sec) * CFG['SR']
                if start + CLIP_SAMPLES > f.frames:
                    start = max(0, f.frames - CLIP_SAMPLES)
                f.seek(start)
                wav = f.read(CLIP_SAMPLES, dtype='float32', always_2d=False)
            if wav.ndim == 2: wav = wav.mean(axis=1)
            wav = wav.astype(np.float32, copy=False)
            if len(wav) > CLIP_SAMPLES: wav = wav[:CLIP_SAMPLES]
            elif len(wav) < CLIP_SAMPLES: wav = np.pad(wav, (0, CLIP_SAMPLES - len(wav)))
            return wav
        except Exception:
            return np.zeros(CLIP_SAMPLES, dtype=np.float32)

    def __getitem__(self, i):
        is_pseudo = (self.pseudo_df is not None and self.is_train and
                     random.random() < self.soundscape_frac)
        if is_pseudo:
            pi = random.randint(0, len(self.pseudo_df) - 1)
            row = self.pseudo_df.iloc[pi]
            wav = self._load_soundscape_window(
                self.soundscape_dir / row['filename'], row['start_sec'])
            y = np.array(row['labels'], dtype=np.float32)
            if CFG['PSEUDO_LABEL_SMOOTH_MIN'] > 0:
                y = np.maximum(y, CFG['PSEUDO_LABEL_SMOOTH_MIN'])
        else:
            fi = i % len(self.focal_df)
            row = self.focal_df.iloc[fi]
            wav = self._load_focal_5s(self.train_audio_dir / row[COL_FILE])
            y = np.zeros(N_CLASSES, dtype=np.float32)
            y[CLS2IDX[row[COL_PRIM]]] = 1.0
            if COL_SECOND in row and isinstance(row[COL_SECOND], str) and row[COL_SECOND]:
                for s in re.split(r"[,;\s]+", row[COL_SECOND].strip('[]').replace("'", '')):
                    if s in CLS2IDX: y[CLS2IDX[s]] = 0.5

        if self.is_train:
            shift = random.randint(-CFG['SR'], CFG['SR'])
            wav = _shift_zero_pad(wav, shift)
            wav = wav * random.uniform(0.7, 1.3)
            wav = self.pink(wav)
            if not is_pseudo:
                wav = self.bg_noise(wav)
        mel_3ch = wav_to_mel_3ch(wav)
        return mel_3ch, torch.from_numpy(y)


In [ ]:
class _SEDB0(nn.Module):
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        self.encoder = timm.create_model(
            CFG['BACKBONE'], pretrained=True, in_chans=3,
            num_classes=0, global_pool='',
            drop_path_rate=CFG['DROP_PATH_RATE'])
        with torch.no_grad():
            dummy = torch.randn(1, 3, CFG['N_MELS'], 313)
            feat_dim = self.encoder(dummy).shape[1]
        self.feat_dim = feat_dim
        self.att = nn.Linear(feat_dim, 1)
        self.fc = nn.Linear(feat_dim, n_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        f = self.encoder(x)
        f = f.mean(dim=2)
        f = f.permute(0, 2, 1)
        att_w = torch.softmax(self.att(f), dim=1)
        pooled = (f * att_w).sum(dim=1)
        return self.fc(self.dropout(pooled))


In [ ]:
def macro_auc(y_true, y_pred):
    y_bin = (y_true >= 0.99).astype(np.int32)
    keep = y_bin.sum(axis=0) > 0
    if keep.sum() == 0: return 0.0
    return roc_auc_score(y_bin[:, keep], y_pred[:, keep], average='macro')

def mixup_data(x, y, alpha):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], lam * y + (1 - lam) * y[idx]

def train_one_fold(fold, focal_train_df, focal_val_df, pseudo_df):
    print(f"\n{'='*60}\nFold {fold}  focal_train={len(focal_train_df)}  focal_val={len(focal_val_df)}  pseudo={len(pseudo_df)}")
    sc_dir = BASE / 'train_soundscapes'
    tr_ds = MixedSEDDataset(focal_train_df, pseudo_df, BASE / 'train_audio', sc_dir, is_train=True)
    va_ds = MixedSEDDataset(focal_val_df,   None,      BASE / 'train_audio', sc_dir, is_train=False)
    tr_dl = DataLoader(tr_ds, CFG['BATCH_SIZE'], shuffle=True,
                       num_workers=CFG['NUM_WORKERS'], pin_memory=True,
                       drop_last=True, persistent_workers=CFG['NUM_WORKERS'] > 0)
    va_dl = DataLoader(va_ds, CFG['BATCH_SIZE'], shuffle=False,
                       num_workers=CFG['NUM_WORKERS'], pin_memory=True,
                       persistent_workers=CFG['NUM_WORKERS'] > 0)

    model = _SEDB0(N_CLASSES).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['LR'], weight_decay=CFG['WD'])
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=CFG['LR'], total_steps=len(tr_dl) * CFG['EPOCHS'],
        pct_start=0.1, anneal_strategy='cos')
    scaler = torch.cuda.amp.GradScaler()
    bce = nn.BCEWithLogitsLoss()

    best_auc, best_state = 0.0, None
    for epoch in range(CFG['EPOCHS']):
        model.train()
        tloss = 0.0; n = 0
        t0 = time.time()
        for step, (mel, y) in enumerate(tr_dl):
            mel, y = mel.to(DEVICE), y.to(DEVICE)
            mel = spec_augment(mel)
            if CFG['MIXUP_ALPHA'] > 0 and random.random() < 0.5:
                mel, y = mixup_data(mel, y, CFG['MIXUP_ALPHA'])
            with torch.cuda.amp.autocast():
                logits = model(mel)
                loss = bce(logits, y)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update()
            sched.step()
            tloss += loss.item() * mel.size(0); n += mel.size(0)

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for mel, y in va_dl:
                mel = mel.to(DEVICE)
                with torch.cuda.amp.autocast():
                    logits = model(mel)
                preds.append(torch.sigmoid(logits.float()).cpu().numpy())
                trues.append(y.numpy())
        preds = np.concatenate(preds); trues = np.concatenate(trues)
        auc = macro_auc(trues, preds)
        elapsed = time.time() - t0
        print(f'  epoch {epoch:02d}  loss={tloss/n:.4f}  val_auc={auc:.4f}  ({elapsed:.0f}s)')
        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    out_pt = Path(CFG['OUT_DIR']) / f'sed_b0_jft_soundscape_fold{fold}.pt'
    torch.save(best_state, out_pt)
    print(f'  saved {out_pt}  best_auc={best_auc:.4f}')
    del model, tr_dl, va_dl
    gc.collect(); torch.cuda.empty_cache()
    return best_auc


In [ ]:
aucs = []
for f, (tr_idx, va_idx) in enumerate(folds):
    if f not in CFG['FOLDS_TO_RUN']:
        continue
    tr_df = balance_index(train_csv.iloc[tr_idx])
    va_df = train_csv.iloc[va_idx].reset_index(drop=True)
    print(f'Fold {f}: train rows {len(tr_df)} (balanced from {len(tr_idx)}), val rows {len(va_df)}')
    auc = train_one_fold(f, tr_df, va_df, pseudo_df)
    aucs.append(auc)
print(f'\n========\nFold val AUCs (focal): {aucs}, mean: {np.mean(aucs) if aucs else 0:.4f}')


In [ ]:
import torch.nn as nn

class SigmoidWrap(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, mel): return torch.sigmoid(self.m(mel))

def export_fold(fold):
    pt = Path(CFG['OUT_DIR']) / f'sed_b0_jft_soundscape_fold{fold}.pt'
    if not pt.exists():
        print(f'  fold {fold}: no .pt found, skipping'); return False
    try:
        m = _SEDB0(N_CLASSES)
        m.load_state_dict(torch.load(str(pt), map_location='cpu'))
        m.eval().cpu()
        w = SigmoidWrap(m).eval().cpu()
        dummy = torch.randn(1, 3, CFG['N_MELS'], 313)
        out = Path(CFG['OUT_DIR']) / f'sed_b0_jft_soundscape_fold{fold}.onnx'
        torch.onnx.export(
            w, dummy, str(out),
            input_names=['mel'], output_names=['prob'],
            dynamic_axes={'mel': {0: 'B'}, 'prob': {0: 'B'}},
            opset_version=17, dynamo=False)
        print(f'  exported {out}  ({out.stat().st_size/1e6:.1f} MB)')
        return True
    except Exception as e:
        print(f'  fold {fold} ONNX FAILED: {e}')
        return False

for f in range(CFG['N_FOLDS']):
    export_fold(f)
